In [40]:
import math
import os
import time
import re
import requests
import polars as pl

from decorator import append
from dotenv import load_dotenv
from duckdb.experimental.spark.sql.functions import length
from datetime import datetime
from utils.postgres.postgres_connection import PostgresConnection


load_dotenv(override=True)

RIOT_PERSONAL_API_KEY = os.getenv("LOL_PERSONAL_API_KEY")
POSTGRES_LOCAL_URL = os.getenv("POSTGRES_LOCAL_URL")
connection_wrapper = PostgresConnection(POSTGRES_LOCAL_URL)

opening connection


In [2]:
_camel_case_regex = re.compile(
    r"(?<=[a-z])(?=[A-Z])"  # camelCase: survive|Min
    r"|(?<=[A-Z])(?=[A-Z][a-z])"  # acronym: HTML|Parser
    r"|(?<=[a-zA-Z])(?=\d)"  # letter→digit: survive|15
    r"|(?<=\d)(?=[a-zA-Z])"  # digit→letter: 15|min
)


def camel_to_snake(input_str: str) -> str:
    return _camel_case_regex.sub("_", input_str).lower()


def snake_all_keys(json_obj: dict):
    if isinstance(json_obj, dict):
        return {camel_to_snake(key): snake_all_keys(value) for key, value in json_obj.items()}

    if isinstance(json_obj, list):
        return [snake_all_keys(value) for value in json_obj]

    return json_obj


In [3]:
players = pl.read_database_uri("SELECT * FROM players", POSTGRES_LOCAL_URL)
players

puuid,game_name,tag_line,region,synced_at
str,str,str,str,datetime[μs]
"""x1jqYLksFHSjj-V_dTeZrzR9fFEVDt…","""bird raven""","""0000""","""EUN1""",2026-05-31 18:10:37.551072
"""QMPPN4oUrY-FW-BBOZ34yDZZmjghPa…","""KaizokuNoOni""","""2829""","""EUN1""",2026-05-31 18:10:37.860377
"""yf1OVyzGtKFWKAh__NIUFtnEisdohM…","""Nolba""","""EUNE""","""EUN1""",2026-05-31 18:10:38.176115
"""gw0udTm33m9cDPfENXndpjlXmkHwxl…","""Defenser""","""EUNE""","""EUN1""",2026-05-31 18:10:38.472933
"""tD8QYcsn9jmPaM7AACN9Id2fwgusfN…","""Dioklis""","""EUNE""","""EUN1""",2026-05-31 18:10:38.757716
…,…,…,…,…
"""dYiEC-ravKg74XNQMp4KsOp4S7aVlR…","""black nails""","""988""","""EUN1""",2026-05-31 19:15:39.289847
"""ZLw_1J27yiyrFC_UN7-Uck-IyYMBLL…","""AleEmotkaZostała""","""RMJ""","""EUN1""",2026-05-31 19:15:39.630180
"""0CPr6RzHVit2UWf0Cu0EXPbIj77eko…","""chiefelo""","""0314""","""EUN1""",2026-05-31 19:15:39.980448


In [29]:
from datetime import date


BASE_EUN1_URL = "https://eun1.api.riotgames.com"

headers = {
    'X-Riot-Token': RIOT_PERSONAL_API_KEY
}

def get_player_challenges(puuid: str):

    player_challanges_url =f"{BASE_EUN1_URL}/lol/challenges/v1/player-data/{puuid}"
    response_challanges = requests.get(player_challanges_url,headers=headers)

    if response_challanges.status_code != 200:
        print("request failed with status code on getting player challanges", response_challanges.status_code)
        return response_challanges.status_code

    response_challanges_json = response_challanges.json()

    all_player_challenges = []

    for challenges_player in response_challanges_json["challenges"]:

        int_achieved_time=challenges_player.get("achievedTime")

        player_challenges = {
           "puuid": puuid,
           "challenge_id":challenges_player.get("challengeId"),
           "percentile":challenges_player.get("percentile"),
           "players_in_level":challenges_player.get("playersInLevel",None),
           "achieved_time":datetime.fromtimestamp(int_achieved_time // 1000) if int_achieved_time else None,
           "value":challenges_player.get("value"),
           "level":challenges_player.get("level"),
           "position":challenges_player.get("position",None),
        }
        all_player_challenges.append(player_challenges)
    return all_player_challenges

get_player_challenges(players["puuid"][0])

[{'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ',
  'challenge_id': 0,
  'percentile': 0.335,
  'players_in_level': None,
  'achieved_time': datetime.datetime(2022, 7, 6, 17, 22, 1),
  'value': 3365.0,
  'level': 'SILVER',
  'position': None},
 {'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ',
  'challenge_id': 1,
  'percentile': 0.306,
  'players_in_level': None,
  'achieved_time': datetime.datetime(2025, 10, 31, 2, 57, 14),
  'value': 300.0,
  'level': 'SILVER',
  'position': None},
 {'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ',
  'challenge_id': 2,
  'percentile': 0.302,
  'players_in_level': None,
  'achieved_time': datetime.datetime(2022, 7, 11, 21, 46, 23),
  'value': 840.0,
  'level': 'SILVER',
  'position': None},
 {'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ',
  'challenge_id': 3,
  'percentile': 0.299,


In [37]:
player_challenges_arr = []

current_index = 0
for player in players.iter_rows(named=True):
    current_index += 1
    if current_index % 50 == 0:
        print(f"progress {math.floor(current_index / len(players) * 100)}%")
    try:
        player_challenges = get_player_challenges(player["puuid"])
        while player_challenges == 429:
            print("retrying after getting rate limited")
            time.sleep(121)
            player_challenges = get_player_challenges(player["puuid"])

        player_challenges_arr.extend(player_challenges)

    except Exception as e:
        print(f"Error fetching snapshot for {player['puuid']}: {e}")
        continue

player_challenges_arr


progress 1%
progress 3%
request failed with status code on getting player challanges 429
retrying after getting rate limited
progress 4%
progress 6%
request failed with status code on getting player challanges 429
retrying after getting rate limited
progress 8%
progress 9%
request failed with status code on getting player challanges 429
retrying after getting rate limited
progress 11%
progress 12%
request failed with status code on getting player challanges 429
retrying after getting rate limited
progress 14%
progress 16%
request failed with status code on getting player challanges 429
retrying after getting rate limited
progress 17%
progress 19%
request failed with status code on getting player challanges 429
retrying after getting rate limited
progress 20%
progress 22%
request failed with status code on getting player challanges 429
retrying after getting rate limited
progress 24%
progress 25%
request failed with status code on getting player challanges 429
retrying after getting rat

[{'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ',
  'challenge_id': 0,
  'percentile': 0.335,
  'players_in_level': None,
  'achieved_time': datetime.datetime(2022, 7, 6, 17, 22, 1),
  'value': 3365.0,
  'level': 'SILVER',
  'position': None},
 {'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ',
  'challenge_id': 1,
  'percentile': 0.306,
  'players_in_level': None,
  'achieved_time': datetime.datetime(2025, 10, 31, 2, 57, 14),
  'value': 300.0,
  'level': 'SILVER',
  'position': None},
 {'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ',
  'challenge_id': 2,
  'percentile': 0.302,
  'players_in_level': None,
  'achieved_time': datetime.datetime(2022, 7, 11, 21, 46, 23),
  'value': 840.0,
  'level': 'SILVER',
  'position': None},
 {'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ',
  'challenge_id': 3,
  'percentile': 0.299,


In [38]:
player_challenges_df=pl.DataFrame(player_challenges_arr)
player_challenges_df

puuid,challenge_id,percentile,players_in_level,achieved_time,value,level,position
str,i64,f64,i64,datetime[μs],f64,str,i64
"""x1jqYLksFHSjj-V_dTeZrzR9fFEVDt…",0,0.335,null,2022-07-06 17:22:01,3365.0,"""SILVER""",null
"""x1jqYLksFHSjj-V_dTeZrzR9fFEVDt…",1,0.306,null,2025-10-31 02:57:14,300.0,"""SILVER""",null
"""x1jqYLksFHSjj-V_dTeZrzR9fFEVDt…",2,0.302,null,2022-07-11 21:46:23,840.0,"""SILVER""",null
"""x1jqYLksFHSjj-V_dTeZrzR9fFEVDt…",3,0.299,null,2022-07-03 21:14:32,805.0,"""SILVER""",null
"""x1jqYLksFHSjj-V_dTeZrzR9fFEVDt…",4,0.345,null,2022-07-23 02:57:28,695.0,"""SILVER""",null
…,…,…,…,…,…,…,…
"""eMwumb2gCUrYSrSf3D1t_9CyIv3kPH…",402403,0.271,null,2026-06-06 00:44:26,14.0,"""IRON""",null
"""eMwumb2gCUrYSrSf3D1t_9CyIv3kPH…",402400,0.157,null,2026-05-09 02:24:52,270.0,"""PLATINUM""",null
"""eMwumb2gCUrYSrSf3D1t_9CyIv3kPH…",402401,0.098,null,2026-05-23 13:19:38,1494.0,"""DIAMOND""",null


In [41]:
connection_wrapper.upsert_dataframe_dto_safe(player_challenges_df, "challenges", ["puuid","challenge_id"])


0